# 03B — Impute Annual Household Income

**Purpose.** Preserve the original `income10k`; add `income10k_imputed`, `income_missing_indicator`, and `income_imputation_method` to the Step 03 annual covariates. No trip-level data are changed.

**Priority rule:** If household income is missing **and** the aligned household employment classification explicitly shows *unemployed*, impute `0` before any other method. Do **not** infer unemployed status from `getting_job`, `other_unemployment`, `retirement`, or the absence of an employment event. A retired household is not automatically unemployed for this rule.

For all other missing income, interpolate within household between observed years; at boundaries carry the nearest observed value; if no household income exists, use observed-income median for that year (overall median fallback). Add method-level audits and preserve original rows.

**Assumption:** Zero income for unemployed households is researcher-imposed, not an observed NIQ value. Employment can coexist with nonemployment income; assess this assumption in robustness checks. **Temporal holdout:** interpolation using future years is not leakage-free for forecast evaluation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = Path("C:/Users/rwen13/Documents/healthy_purchase_data")
MASTER_ROOT = DATA_ROOT / "processed" / "hmm_panel" / "hmm_master"
INPUT_FILE = MASTER_ROOT / "hmm_annual_covariates.parquet"
OUTPUT_FILE = MASTER_ROOT / "hmm_annual_covariates_income_imputed.parquet"
AUDIT_DIR = MASTER_ROOT / "validation"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Run Step 03 first: {INPUT_FILE}")

annual = pd.read_parquet(INPUT_FILE)
required = {"hhid", "panel_year", "income10k"}
missing_columns = required - set(annual.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

# Household IDs are identifiers and must never be cast to numeric.
annual["hhid"] = annual["hhid"].astype("string")
annual["panel_year"] = pd.to_numeric(annual["panel_year"], errors="raise").astype("int16")
annual["income10k"] = pd.to_numeric(annual["income10k"], errors="coerce")
if annual[["hhid", "panel_year"]].isna().any().any():
    raise ValueError("Null household/year key in annual covariates")
if annual.duplicated(["hhid", "panel_year"]).any():
    raise ValueError("Expected one annual row per (hhid, panel_year)")
if annual["income10k"].notna().sum() == 0:
    raise ValueError("No observed income values available for imputation")

print(f"Input household-years: {len(annual):,}")
print(f"Input unique households: {annual['hhid'].nunique():,}")
print(f"Missing original income: {annual['income10k'].isna().sum():,}")

In [ ]:
# Load aligned household employment information from the demographic panel.
# The Step 03 compact annual file generally does not contain household employment status.
# IMPORTANT: set UNEMPLOYED_COLUMN to an actual binary variable in the demographic TSV:
# 1 = unemployed household, 0 = not unemployed, NA = unknown.
# Do NOT use an annual transition event flag in place of a current employment state.
DEMOGRAPHICS_FILE = DATA_ROOT / "processed/demographics/household_demographics_with_events_2008_2017.tsv"
UNEMPLOYED_COLUMN = None  # Replace with the VERIFIED 0/1 household-unemployed column name.

if not DEMOGRAPHICS_FILE.is_file():
    raise FileNotFoundError(f"Missing demographics: {DEMOGRAPHICS_FILE}")

# Inspect header without loading the large file if status column is not configured.
available_columns = pd.read_csv(DEMOGRAPHICS_FILE, sep="\t", nrows=0).columns.tolist()
if UNEMPLOYED_COLUMN is None or UNEMPLOYED_COLUMN not in available_columns:
    potential = [c for c in available_columns if any(t in c.lower() for t in ("employ", "work", "retir", "job", "labor"))]
    raise ValueError(
        "Set UNEMPLOYED_COLUMN to a VERIFIED current household unemployment flag (0/1) "
        "in the demographic source; do not use life-event flags. "
        f"Candidate columns: {potential}"
    )

employment = pd.read_csv(
    DEMOGRAPHICS_FILE, sep="\t", usecols=["hhid", "panel_year", UNEMPLOYED_COLUMN],
    dtype={"hhid": "string"}, low_memory=False
)
employment["panel_year"] = pd.to_numeric(employment["panel_year"], errors="raise").astype("int16")
employment[UNEMPLOYED_COLUMN] = pd.to_numeric(employment[UNEMPLOYED_COLUMN], errors="coerce")
if not employment[UNEMPLOYED_COLUMN].dropna().isin([0, 1]).all():
    raise ValueError("Unemployment flag must contain only 0, 1, or missing values")
if employment.duplicated(["hhid", "panel_year"]).any():
    raise ValueError("Duplicate household-year keys in aligned employment file")

annual = annual.merge(
    employment.rename(columns={UNEMPLOYED_COLUMN: "_household_unemployed"}),
    on=["hhid", "panel_year"], how="left", validate="one_to_one", sort=False
)
print("Unemployment status available:", annual["_household_unemployed"].notna().sum())
print("Missing income AND unemployed:",
      (annual["income10k"].isna() & annual["_household_unemployed"].eq(1)).sum())


In [ ]:
# Sort only for within-household time interpolation; restore source row order later.
work = annual.copy()
work["_original_row"] = np.arange(len(work), dtype=np.int64)
work = work.sort_values(["hhid", "panel_year"], kind="stable").reset_index(drop=True)
original_missing = work["income10k"].isna()
work["income_missing_indicator"] = original_missing.astype("int8")

# Data are annual, so interpolate using the actual calendar year, not the row position.
def interpolate_household(group: pd.DataFrame) -> pd.Series:
    observed = group["income10k"].notna()
    result = pd.Series(np.nan, index=group.index, dtype="float64")
    if observed.any():
        x_observed = group.loc[observed, "panel_year"].to_numpy(dtype=float)
        y_observed = group.loc[observed, "income10k"].to_numpy(dtype=float)
        x_all = group["panel_year"].to_numpy(dtype=float)
        # np.interp linearly interpolates inside; carries nearest endpoints outside.
        result[:] = np.interp(x_all, x_observed, y_observed)
    return result

# Apply the unemployment-zero rule BEFORE interpolating other gaps.
# Zero values introduced by this rule are treated as anchors only for the same year,
# not as independently observed income values for other years.
unemployed_missing = original_missing & work["_household_unemployed"].eq(1).fillna(False)

work["income10k_imputed"] = (
    work.groupby("hhid", sort=False, group_keys=False)
        .apply(interpolate_household, include_groups=False)
        .reindex(work.index)
)

# Explicit override for missing-income, currently unemployed household-years.
work.loc[unemployed_missing, "income10k_imputed"] = 0.0

# Mark interior gaps separately from leading/trailing gaps.
observed_year = work["panel_year"].where(work["income10k"].notna())
first_obs = observed_year.groupby(work["hhid"]).transform("min")
last_obs = observed_year.groupby(work["hhid"]).transform("max")
work["income_imputation_method"] = "observed"
filled_from_household = original_missing & work["income10k_imputed"].notna()
work.loc[filled_from_household & (work["panel_year"] < first_obs), "income_imputation_method"] = "carry_backward"
work.loc[filled_from_household & (work["panel_year"] > last_obs), "income_imputation_method"] = "carry_forward"
work.loc[filled_from_household & work["panel_year"].between(first_obs, last_obs), "income_imputation_method"] = "linear_interpolation"

work.loc[unemployed_missing, "income_imputation_method"] = "unemployed_zero"

# Families without observed income in ANY available year: yearly observed median.
median_by_year = work.groupby("panel_year")["income10k"].median()
overall_median = work["income10k"].median()
fully_missing = work["income10k_imputed"].isna()
work.loc[fully_missing, "income10k_imputed"] = (
    work.loc[fully_missing, "panel_year"].map(median_by_year)
)
work.loc[fully_missing, "income_imputation_method"] = "year_median"
still_missing = work["income10k_imputed"].isna()
work.loc[still_missing, "income10k_imputed"] = overall_median
work.loc[still_missing, "income_imputation_method"] = "overall_median"

# Observed income always stays identical in both original and completed series.
assert np.allclose(
    work.loc[~original_missing, "income10k"],
    work.loc[~original_missing, "income10k_imputed"],
    equal_nan=True,
)
assert work["income10k_imputed"].notna().all()
assert work.loc[unemployed_missing, "income10k_imputed"].eq(0).all()
assert work.loc[unemployed_missing, "income_imputation_method"].eq("unemployed_zero").all()
assert (work["income_missing_indicator"] == work["income10k"].isna().astype("int8")).all()

# Keep the annual rows and all original columns in the original order.
result = (work.sort_values("_original_row", kind="stable")
              .drop(columns=["_original_row", "_household_unemployed"])
              .reset_index(drop=True))
assert len(result) == len(annual)
assert result[["hhid", "panel_year"]].equals(annual[["hhid", "panel_year"]].reset_index(drop=True))
print(result["income_imputation_method"].value_counts().to_string())

In [ ]:
# Save a NEW annual file rather than overwriting the Step 03 source.
result.to_parquet(OUTPUT_FILE, index=False)

by_year = (result.groupby("panel_year", as_index=False)
    .agg(annual_rows=("hhid", "size"),
         observed_income=("income_missing_indicator", lambda v: int((v == 0).sum())),
         missing_original=("income_missing_indicator", "sum"),
         missing_after=("income10k_imputed", lambda v: int(v.isna().sum()))))
by_year["original_missing_rate"] = by_year["missing_original"] / by_year["annual_rows"]
by_year.to_csv(AUDIT_DIR / "income_imputation_by_year.csv", index=False)

by_method = (result["income_imputation_method"].value_counts(dropna=False)
    .rename_axis("method").reset_index(name="household_years"))
by_method.to_csv(AUDIT_DIR / "income_imputation_by_method.csv", index=False)

by_household = (result.groupby("hhid")["income_missing_indicator"]
    .agg(missing_years="sum", observed_years=lambda s: int((s == 0).sum())))
summary = pd.DataFrame([{
    "households": len(by_household),
    "households_any_missing": int((by_household.missing_years > 0).sum()),
    "households_all_missing": int((by_household.observed_years == 0).sum()),
    "households_partially_missing": int(((by_household.missing_years > 0) & (by_household.observed_years > 0)).sum()),
    "original_missing_household_years": int(result.income_missing_indicator.sum()),
    "remaining_missing_household_years": int(result.income10k_imputed.isna().sum()),
}])
summary.to_csv(AUDIT_DIR / "income_imputation_summary.csv", index=False)

print("Saved:", OUTPUT_FILE)
print("\nHousehold summary:")
print(summary.to_string(index=False))
print("\nBy year:")
print(by_year.to_string(index=False))
print("\nBy method:")
print(by_method.to_string(index=False))

## Modeling note

Use `income10k_imputed` with the original `income_missing_indicator` as a possible control. The additional method category `unemployed_zero` identifies values imposed by the unemployment rule. Neither `other_unemployment` (an event) nor `retirement` (an event) establishes *current unemployment* by itself.

For temporal forecasting holdouts, refit imputation using training-period information only; do not leak future income into the training sample.

This notebook writes a NEW annual file and leaves all Step 03 trip files unchanged. Point subsequent model loaders to `hmm_annual_covariates_income_imputed.parquet` explicitly.
